# SAGDA AUGUMENTATION 
## Introduction

This notebook outlines the approach for merging and augmenting agricultural datasets to enhance credit scoring models for African farmers.

## What is SAGDA ?
* SAGDA stands for Synthetic Agricultural and Geo-Environmental Data Augmentation. 
* It is a method or tool used to generate or enhance agricultural and environmental variables—such as crop yields, rainfall, soil quality, fertilizer use, and climate factors—by simulating realistic data conditioned on existing farmer profiles. 
* This augmentation helps create richer datasets that better reflect the real-world agricultural context, particularly for regions like Africa, and improves the accuracy and robustness of analyses such as credit scoring for farmers.

## Farmer IDs for Merging
* We utilize the `farmer_id` column created in the preprocessing script `lsms_missing_data_cleaning.ipynb`. 
* This ID is deterministic, constructed from a combination of country, survey wave, season, and household merge ID (`hh_id_merge`). Because of this design, the `farmer_id` remains consistent throughout all processing steps. 
* Consequently, the SAGDA output will use the same `farmer_id` values as the LSMS minimum viable product (MVP), ensuring seamless merging.
* After generating the synthetic SAGDA data , we'll additionally merge it qith the `lsms_final_mvp.csv dataset` 

## SAGDA Feature Augmentation and Importance
SAGDA generates realistic agricultural and environmental variables tailored to the African context, which are crucial for accurate credit scoring. These features include:

- **Yield-related Variables:**  
  - `yield_kg_ha`, `yield_stability`, `expected_yield`  
  These serve as some of the strongest predictors of a farmer's repayment capacity.
  
- **Climate and Weather Variables:**  
  - `rainfall_mm`, `drought_risk`, `temperature`, `soil_moisture`  
  These explain seasonal default risks, which are particularly significant in East Africa.

- **Soil and Input Variables:**  
  - `soil_npk`, `fertilizer_recommendation`, `soil_quality_index`  
  These reflect the farmer's professionalism and investment in inputs.

- **Risk Indicators:**  
  - `flood_risk`, `pest_pressure`, `climate_stress_index`

These SAGDA-derived features complement existing LSMS columns such as `farm_size`, `irrigated`, `inorganic_fertilizer`, `livestock`, and `shocks`. Together, they enable the creation of powerful derived signals, for example, `input_efficiency` calculated as fertilizer applied divided by expected yield, enhancing the predictive power of credit scoring models.


In [1]:
# import needed dependancies 

import pandas as pd
pd.options.display.max_columns = None
pd.options.display.max_rows = None

import numpy as np
from sagda import generate, augment
import os
from datetime import datetime


# get the current working directory

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)


# get the path to the data folder
data_path = os.path.join(
    parent_dir,
    "Data sets"
)

In [ ]:
# Load the filtered LSMS MVP dataset from previous step

lsms_final_mvp = pd.read_csv(os.path.join(data_path , "Clean data/lsms_final_mvp.csv"))

In [ ]:
#lsms_final_mvp.info

In [4]:
lsms_final_mvp.shape

(32000, 40)

In [5]:
print(lsms_final_mvp.head(3))

      last_col_name   country  wave  season           pw  strataid urban  \
0  fc0d9d525b289396  Tanzania   4.0     1.0  5015.339844         8    No   
1  a14771d719826178  Tanzania   1.0     1.0  3934.863770         7    No   
2  0fd194d7b10b99fa  Tanzania   1.0     1.0   631.631348        15    No   

      hh_id_merge  hh_id_obs  hh_size  hh_dependency_ratio  age_manager  \
0        4994-001  6007574.0      5.0                  1.5         33.0   
1  18010270010755  6004480.0      1.0                  0.0         24.0   
2  53010190110075  6007920.0      4.0                  0.0         64.0   

   hh_primary_education  hh_formal_education  farm_work  farm_size plot_owned  \
0                   0.0                  1.0        1.0   1.444729         No   
1                   0.0                  1.0        1.0   0.294600        Yes   
2                   1.0                  1.0        1.0   0.367100         No   

  irrigated inorganic_fertilizer            main_crop livestock  hh_s

In [6]:
print(lsms_final_mvp.columns)

Index(['last_col_name', 'country', 'wave', 'season', 'pw', 'strataid', 'urban',
       'hh_id_merge', 'hh_id_obs', 'hh_size', 'hh_dependency_ratio',
       'age_manager', 'hh_primary_education', 'hh_formal_education',
       'farm_work', 'farm_size', 'plot_owned', 'irrigated',
       'inorganic_fertilizer', 'main_crop', 'livestock', 'hh_shock',
       'crop_shock', 'drought_shock', 'dist_market', 'pw_missing',
       'strataid_missing', 'hh_primary_education_missing', 'hh_shock_missing',
       'age_manager_missing', 'plot_owned_missing', 'irrigated_missing',
       'inorganic_fertilizer_missing', 'crop_shock_missing',
       'farm_size_missing', 'main_crop_missing', 'livestock_missing',
       'drought_shock_missing', 'drought_shock_no_coverage',
       'dist_market_missing'],
      dtype='object')


In [7]:
lsms_final_mvp.rename(columns={'last_col_name' : 'farmer_id'} , inplace = True)

In [8]:
lsms_final_mvp.columns

Index(['farmer_id', 'country', 'wave', 'season', 'pw', 'strataid', 'urban',
       'hh_id_merge', 'hh_id_obs', 'hh_size', 'hh_dependency_ratio',
       'age_manager', 'hh_primary_education', 'hh_formal_education',
       'farm_work', 'farm_size', 'plot_owned', 'irrigated',
       'inorganic_fertilizer', 'main_crop', 'livestock', 'hh_shock',
       'crop_shock', 'drought_shock', 'dist_market', 'pw_missing',
       'strataid_missing', 'hh_primary_education_missing', 'hh_shock_missing',
       'age_manager_missing', 'plot_owned_missing', 'irrigated_missing',
       'inorganic_fertilizer_missing', 'crop_shock_missing',
       'farm_size_missing', 'main_crop_missing', 'livestock_missing',
       'drought_shock_missing', 'drought_shock_no_coverage',
       'dist_market_missing'],
      dtype='object')

In [9]:
print(f"Unique farmers: {lsms_final_mvp['farmer_id'].nunique():,}\n")
print(lsms_final_mvp[['farmer_id', 'country', 'wave', 'farm_size', 'irrigated', 'inorganic_fertilizer', 'livestock']].head())

Unique farmers: 11,408

          farmer_id   country  wave  farm_size irrigated inorganic_fertilizer  \
0  fc0d9d525b289396  Tanzania   4.0   1.444729        No                   No   
1  a14771d719826178  Tanzania   1.0   0.294600        No                   No   
2  0fd194d7b10b99fa  Tanzania   1.0   0.367100        No                   No   
3  387a5b0b8249726b  Tanzania   5.0   0.052609        No                   No   
4  5760003d7898ce8a  Tanzania   4.0   0.428967        No                   No   

  livestock  
0       Yes  
1        No  
2       Yes  
3       Yes  
4       Yes  


### SAGDA Implementation (Generation + Augmentation)

In [10]:
print("lsms final length count" ,len( lsms_final_mvp))


lsms final length count 32000


In [11]:
# ENHANCED CUSTOM SAGDA GENERATOR 


n = len(lsms_final_mvp)

np.random.seed(42)

sagda_df = pd.DataFrame({
    'farmer_id': lsms_final_mvp['farmer_id'].values,
    
    # Temporal
    'date': pd.date_range(start="2024-03-01", periods=n, freq='D')[:n],   # Planting season focus
    
    # Yield & Productivity
    'yield_kg_ha': np.random.normal(2800, 950, n).clip(600, 7200),
    'yield_stability': np.random.uniform(0.45, 0.95, n),           # Lower = more volatile
    'expected_yield_next_season': np.random.normal(2900, 1000, n).clip(700, 7500),
    
    # Climate & Weather
    'rainfall_mm': np.random.normal(680, 190, n).clip(180, 1350),
    'rainfall_variability': np.random.uniform(0.15, 0.65, n),
    'temperature_avg': np.random.normal(24.8, 3.8, n).clip(17, 33),
    'soil_moisture': np.random.uniform(0.35, 0.88, n),
    
    # Soil & Nutrients
    'soil_npk': np.random.uniform(35, 88, n),
    'soil_quality_index': np.random.uniform(0.42, 0.94, n),
    'soil_ph': np.random.uniform(5.2, 7.8, n),
    
    # Risk Factors
    'drought_risk': np.random.beta(2.2, 5.5, n),          # Most farmers have moderate-low risk
    'flood_risk': np.random.beta(1.4, 6.5, n),
    'pest_pressure': np.random.uniform(0.18, 0.82, n),
    'climate_stress_index': np.random.uniform(0.12, 0.78, n),
    
    # Input & Management
    'fertilizer_recommendation_kg': np.random.normal(180, 70, n).clip(40, 450),
    'irrigation_suitability': np.random.uniform(0.3, 0.95, n),
    'crop_health_index': np.random.uniform(0.55, 0.96, n),
    
    # Market & Economic
    'market_access_score': np.random.uniform(0.4, 0.92, n),
    'price_volatility_index': np.random.uniform(0.2, 0.75, n)
})



In [12]:
print(f" Enhanced SAGDA DataFrame created with shape: {sagda_df.shape}")
print("Features included:", sagda_df.columns.tolist())

 Enhanced SAGDA DataFrame created with shape: (32000, 21)
Features included: ['farmer_id', 'date', 'yield_kg_ha', 'yield_stability', 'expected_yield_next_season', 'rainfall_mm', 'rainfall_variability', 'temperature_avg', 'soil_moisture', 'soil_npk', 'soil_quality_index', 'soil_ph', 'drought_risk', 'flood_risk', 'pest_pressure', 'climate_stress_index', 'fertilizer_recommendation_kg', 'irrigation_suitability', 'crop_health_index', 'market_access_score', 'price_volatility_index']


#### FUTURE FEATURE ADDITION:
 * Adding  a NASA DATA API key , so that the library’s behavior matches the NASA POWER API requirement for a valid key when making request

### Save the Sagda generated dataset

In [13]:
output_path = os.path.join(data_path , "Processed data/SAGDA_generated_data.csv")
sagda_df.to_csv(output_path, index=False)
print(f' SAGDA generated  dataset saved to: {output_path}')
print(f'  Rows: {len(sagda_df):,} | Columns: {sagda_df.shape[1]}')

 SAGDA generated  dataset saved to: e:\DSF\flatiron\phase5\agriscoreproject\Data sets\Processed data/SAGDA_generated_data.csv
  Rows: 32,000 | Columns: 21


### Merge the LSMS final mvp dataset with the generated sagda dataset

In [14]:
print("Before merge:")
print(f"LSMS MVP rows: {lsms_final_mvp.shape[0]:,}, Unique farmers: {lsms_final_mvp['farmer_id'].nunique():,}")
print(f"SAGDA rows: {sagda_df.shape[0]:,}, Unique farmers: {sagda_df['farmer_id'].nunique():,}")

# Aggregate SAGDA to 1 row per farmer (Important!)
sagda_agg = sagda_df.groupby('farmer_id').mean(numeric_only=True).reset_index()

# Merge with LSMS
final_merged = lsms_final_mvp.merge(
    sagda_agg, 
    on='farmer_id', 
    how='left'
)

print("\n")
print("After merge:")
print(f"Final Merged Shape: {final_merged.shape} ")
print(f"Unique farmers: {final_merged['farmer_id'].nunique():,}")



Before merge:
LSMS MVP rows: 32,000, Unique farmers: 11,408
SAGDA rows: 32,000, Unique farmers: 11,408


After merge:
Final Merged Shape: (32000, 59) 
Unique farmers: 11,408


### Validation & Derived Features

In [15]:
# TYPE CONVERSION & DERIVED FEATURES 

# Convert key columns to numeric (force errors to NaN)

numeric_cols = ['inorganic_fertilizer', 'farm_size', 'livestock', 
                'yield_kg_ha', 'rainfall_mm', 'soil_npk', 'soil_quality_index']

for col in numeric_cols:
    if col in final_merged.columns:
        final_merged[col] = pd.to_numeric(final_merged[col], errors='coerce')

print(" Numeric conversion completed")


 Numeric conversion completed


In [16]:

# Now create derived features safely
final_merged['yield_per_ha'] = final_merged.get('yield_kg_ha', 
                            final_merged.get('farm_size', 1) * 800)  # fallback

final_merged['input_efficiency'] = final_merged['inorganic_fertilizer'] / \
                                   (final_merged['yield_kg_ha'] + 1)

final_merged['climate_risk_score'] = (
    final_merged.get('drought_risk', 0) + 
    final_merged.get('flood_risk', 0)
) / 2

final_merged['fertilizer_per_ha'] = final_merged['inorganic_fertilizer'] / \
                                    (final_merged['farm_size'] + 0.1)

print("\n Derived features created successfully")

# Handle any remaining infinities or NaNs
final_merged.replace([np.inf, -np.inf], np.nan, inplace=True)




 Derived features created successfully


In [17]:
#  VALIDATION 
print("\nKey Credit Features Summary:")
cols_to_show = ['farm_size', 'irrigated', 'inorganic_fertilizer', 'livestock',
                'yield_kg_ha', 'rainfall_mm', 'input_efficiency', 'climate_risk_score']

print(final_merged[cols_to_show].describe().round(2))

print("\nMissing values in key columns:")
print(final_merged[cols_to_show].isnull().sum())


Key Credit Features Summary:
       farm_size  inorganic_fertilizer  livestock  yield_kg_ha  rainfall_mm  \
count   32000.00                 627.0        0.0     32000.00     32000.00   
mean        6.74                   0.0        NaN      2803.43       680.77   
std       269.45                   0.0        NaN       562.03       111.71   
min         0.00                   0.0        NaN       600.00       180.00   
25%         0.62                   0.0        NaN      2471.71       617.15   
50%         1.46                   0.0        NaN      2799.95       680.99   
75%         3.16                   0.0        NaN      3132.69       745.93   
max     20977.71                   0.0        NaN      6529.93      1348.92   

       input_efficiency  climate_risk_score  
count             627.0            32000.00  
mean                0.0                0.23  
std                 0.0                0.06  
min                 0.0                0.02  
25%                 0.0     

### Confirm augumented dataset and Save 

In [18]:
lsms_final_mvp.shape

(32000, 40)

In [19]:
print(f"Unique farmers: {lsms_final_mvp['farmer_id'].nunique():,}")

Unique farmers: 11,408


In [20]:
final_merged.shape

(32000, 63)

In [21]:
print(f"Unique farmers: {final_merged['farmer_id'].nunique():,}")

Unique farmers: 11,408


In [22]:
print(final_merged.columns)

Index(['farmer_id', 'country', 'wave', 'season', 'pw', 'strataid', 'urban',
       'hh_id_merge', 'hh_id_obs', 'hh_size', 'hh_dependency_ratio',
       'age_manager', 'hh_primary_education', 'hh_formal_education',
       'farm_work', 'farm_size', 'plot_owned', 'irrigated',
       'inorganic_fertilizer', 'main_crop', 'livestock', 'hh_shock',
       'crop_shock', 'drought_shock', 'dist_market', 'pw_missing',
       'strataid_missing', 'hh_primary_education_missing', 'hh_shock_missing',
       'age_manager_missing', 'plot_owned_missing', 'irrigated_missing',
       'inorganic_fertilizer_missing', 'crop_shock_missing',
       'farm_size_missing', 'main_crop_missing', 'livestock_missing',
       'drought_shock_missing', 'drought_shock_no_coverage',
       'dist_market_missing', 'yield_kg_ha', 'yield_stability',
       'expected_yield_next_season', 'rainfall_mm', 'rainfall_variability',
       'temperature_avg', 'soil_moisture', 'soil_npk', 'soil_quality_index',
       'soil_ph', 'drought_ri

####  Save the merged data set 

In [23]:
output_path = os.path.join(data_path , "Processed data/augumented_data.csv")
final_merged.to_csv(output_path, index=False)
print(f' Filtered mvp dataset saved to: {output_path}')
print(f'  Rows: {len(final_merged):,} | Columns: {final_merged.shape[1]}')

 Filtered mvp dataset saved to: e:\DSF\flatiron\phase5\agriscoreproject\Data sets\Processed data/augumented_data.csv
  Rows: 32,000 | Columns: 63
